<a href="https://colab.research.google.com/github/SanjaraT/Bengali-Sentiment-Analysis-using-Transformer-Models/blob/main/bengali_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers datasets accelerate scikit-learn scipy

In [5]:
from google.colab import drive; drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import time

import numpy as np
import pandas as pd

In [7]:
POS_PATH = "/content/drive/MyDrive/bengali_senti_dataset/all_positive_8500.txt"
NEG_PATH = "/content/drive/MyDrive/bengali_senti_dataset/all_negative_3307.txt"

OUT = "results"
MAX_LEN = 128                        # raise to the p95 printed by check_lengths() if it is larger
EPOCHS, LR, BATCH = 3, 1e-5, 16
SEEDS = (1, 2, 3)
MODELS = {
    "mBERT": "bert-base-multilingual-cased",
    "DistilmBERT": "distilbert-base-multilingual-cased",
    "XLM-R": "xlm-roberta-base",
}
os.makedirs(OUT, exist_ok=True)

# split (de-duplicated, stratified, seeded) saved to disk

In [9]:
def prepare_data():
    from sklearn.model_selection import train_test_split

    os.makedirs(OUT, exist_ok=True)

    def read(path):
        return [line.strip() for line in open(path, encoding="utf-8") if line.strip()]

    pos, neg = read(POS_PATH), read(NEG_PATH)
    df = pd.DataFrame({"text": pos + neg, "label": [1] * len(pos) + [0] * len(neg)})
    print("raw:", len(df), df.label.value_counts().to_dict())

    n_labels = df.groupby("text").label.nunique()
    conflicting = n_labels[n_labels > 1].index
    print("exact duplicate texts:", int(df.duplicated("text").sum()),
          "| texts that appear with BOTH labels:", len(conflicting))
    df = df[~df.text.isin(conflicting)].drop_duplicates("text").reset_index(drop=True)
    print("after cleaning:", len(df), df.label.value_counts().to_dict())

    train_df, test_df = train_test_split(df, test_size=0.1, stratify=df.label, random_state=42)
    train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
    train_df.to_csv(f"{OUT}/train.csv", index=False)
    test_df.to_csv(f"{OUT}/test.csv", index=False)
    print("train:", len(train_df), "| test:", len(test_df), test_df.label.value_counts().to_dict())
    return train_df, test_df


def check_lengths(train_df):
    """Print token-length percentiles so you can choose MAX_LEN on evidence."""
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODELS["XLM-R"])
    sample = train_df.text.sample(min(2000, len(train_df)), random_state=0)
    lens = [len(tok(t)["input_ids"]) for t in sample]
    p50, p95, p99 = np.percentile(lens, [50, 95, 99])
    print(f"token length p50={p50:.0f}  p95={p95:.0f}  p99={p99:.0f}  (MAX_LEN is {MAX_LEN})")

In [11]:
train_df, test_df = prepare_data()
check_lengths(train_df)

raw: 11807 {1: 8500, 0: 3307}
exact duplicate texts: 1052 | texts that appear with BOTH labels: 1
after cleaning: 10754 {1: 7697, 0: 3057}
train: 9678 | test: 1076 {1: 770, 0: 306}


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


token length p50=18  p95=79  p99=150  (MAX_LEN is 128)


# training / evaluation of one run

In [22]:
def run(name, ckpt, seed, weighted, train_df, test_df, keep_model=False):
    import torch
    from datasets import Dataset
    from sklearn.metrics import accuracy_score, f1_score, recall_score
    from sklearn.utils.class_weight import compute_class_weight
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

    class WeightedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=w)
            return (loss, outputs) if return_outputs else loss

    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(ckpt)
    enc = lambda batch: tok(batch["text"], truncation=True, max_length=MAX_LEN)
    tr = Dataset.from_pandas(train_df).map(enc, batched=True)
    te = Dataset.from_pandas(test_df).map(enc, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)

    weights = None
    if weighted:
        weights = torch.tensor(
            compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df.label.values),
            dtype=torch.float)

    args = TrainingArguments(
        output_dir=f"{OUT}/tmp_{name}_{seed}", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, seed=seed,
        fp16=torch.cuda.is_available(), save_strategy="no", report_to="none")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr, class_weights=weights,
                              data_collator=DataCollatorWithPadding(tok))
    trainer.train()

    preds = trainer.predict(te).predictions.argmax(1)
    y = test_df.label.values
    tag = "weighted" if weighted else "unweighted"
    np.save(f"{OUT}/preds_{name}_{seed}_{tag}.npy", preds)
    row = dict(model=name, seed=seed, weighted=weighted,
               acc=accuracy_score(y, preds), macro_f1=f1_score(y, preds, average="macro"),
               neg_recall=recall_score(y, preds, pos_label=0),
               pos_recall=recall_score(y, preds, pos_label=1))
    if keep_model:
        return row, model, tok
    del model, trainer
    torch.cuda.empty_cache()
    return row


def run_all(train_df, test_df):
    """12 runs: 3 models x 3 seeds (weighted) + XLM-R x 3 seeds UNweighted. Resumable."""
    path = f"{OUT}/runs.csv"
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    plan = [(n, c, s, True) for n, c in MODELS.items() for s in SEEDS]
    plan += [("XLM-R", MODELS["XLM-R"], s, False) for s in SEEDS]
    for name, ckpt, seed, weighted in plan:
        if len(done) and ((done.model == name) & (done.seed == seed) & (done.weighted == weighted)).any():
            print("skip (already done):", name, seed, weighted)
            continue
        t0 = time.time()
        row = run(name, ckpt, seed, weighted, train_df, test_df)
        print(f"{name} seed={seed} weighted={weighted}: acc={row['acc']:.4f} "
              f"macroF1={row['macro_f1']:.4f} negRecall={row['neg_recall']:.4f} ({time.time()-t0:.0f}s)")
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(path, index=False)  # saved after every run, so a Colab disconnect loses nothing
    return done

In [27]:
runs = run_all(train_df, test_df)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/9678 [00:00<?, ? examples/s]

Map:   0%|          | 0/1076 [00:00<?, ? examples/s]

... (76 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.379906
1000,0.199897
1500,0.162765


XLM-R seed=3 weighted=False: acc=0.9526 macroF1=0.9425 negRecall=0.9379 (237s)


#  summary table + exact McNemar test

In [28]:
def mcnemar_p(correct_a, correct_b):
    from scipy.stats import binomtest
    b = int((correct_a & ~correct_b).sum())   # A right, B wrong
    c = int((~correct_a & correct_b).sum())   # A wrong, B right
    return 1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)


def summarize(test_df):
    runs = pd.read_csv(f"{OUT}/runs.csv")
    runs["setting"] = np.where(runs.weighted, "class-weighted loss", "unweighted loss")
    g = runs.groupby(["model", "setting"])[["acc", "macro_f1", "neg_recall"]].agg(["mean", "std"])
    lines = ["| Model | Loss | Accuracy | Macro-F1 | Negative-class recall |", "|---|---|---|---|---|"]
    for (model, setting), r in g.iterrows():
        fmt = lambda k: f"{r[(k, 'mean')]*100:.2f} +/- {r[(k, 'std')]*100:.2f}"
        lines.append(f"| {model} | {setting} | {fmt('acc')} | {fmt('macro_f1')} | {fmt('neg_recall')} |")

    y = test_df.label.values
    lines += ["", "McNemar exact p-values, same test set (p < 0.05 = difference unlikely to be noise):", "",
              "| Comparison (weighted runs) | seed 1 | seed 2 | seed 3 |", "|---|---|---|---|"]
    for a, b in (("XLM-R", "DistilmBERT"), ("XLM-R", "mBERT"), ("mBERT", "DistilmBERT")):
        ps = []
        for s in SEEDS:
            pa = np.load(f"{OUT}/preds_{a}_{s}_weighted.npy") == y
            pb = np.load(f"{OUT}/preds_{b}_{s}_weighted.npy") == y
            ps.append(f"{mcnemar_p(pa, pb):.3f}")
        lines.append(f"| {a} vs {b} | " + " | ".join(ps) + " |")
    text = "\n".join(lines)
    open(f"{OUT}/summary.md", "w", encoding="utf-8").write(text + "\n")
    print(text)

In [29]:
summarize(test_df)

| Model | Loss | Accuracy | Macro-F1 | Negative-class recall |
|---|---|---|---|---|
| DistilmBERT | class-weighted loss | 93.49 +/- 0.19 | 92.05 +/- 0.23 | 89.43 +/- 0.82 |
| XLM-R | class-weighted loss | 95.20 +/- 0.11 | 94.14 +/- 0.15 | 92.59 +/- 1.32 |
| XLM-R | unweighted loss | 95.29 +/- 0.23 | 94.25 +/- 0.27 | 92.70 +/- 0.94 |
| mBERT | class-weighted loss | 94.05 +/- 0.19 | 92.71 +/- 0.25 | 90.09 +/- 2.74 |

McNemar exact p-values, same test set (p < 0.05 = difference unlikely to be noise):

| Comparison (weighted runs) | seed 1 | seed 2 | seed 3 |
|---|---|---|---|
| XLM-R vs DistilmBERT | 0.036 | 0.016 | 0.025 |
| XLM-R vs mBERT | 0.144 | 0.105 | 0.124 |
| mBERT vs DistilmBERT | 0.405 | 0.441 | 0.441 |


# CPU latency of the model

In [30]:
def latency(model_id, test_csv=None, n=200):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    test_csv = test_csv or f"{OUT}/test.csv"

    torch.set_num_threads(2)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).eval()
    texts = pd.read_csv(test_csv).text.sample(n, random_state=0).tolist()
    with torch.no_grad():
        for t in texts[:10]:                                   # warm-up, not timed
            model(**tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN))
        times = []
        for t in texts:
            s = time.perf_counter()
            model(**tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN))
            times.append((time.perf_counter() - s) * 1000)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    out = (f"{model_id}: {params:.0f}M params | p50 {np.percentile(times, 50):.0f} ms | "
           f"p95 {np.percentile(times, 95):.0f} ms | {n} requests, CPU, 2 threads, batch size 1")
    print(out)
    open(f"{OUT}/latency.md", "a", encoding="utf-8").write(out + "\n")

In [13]:
import time, numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
torch.set_num_threads(2)

# Use the already loaded test_df variable instead of reading from disk
texts = test_df.text.sample(300, random_state=0).tolist()

def bench(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).eval()
    with torch.no_grad():
        for t in texts[:30]:                      # longer warm-up
            model(**tok(t, return_tensors="pt", truncation=True, max_length=128))
        ms = []
        for t in texts:
            s = time.perf_counter()
            model(**tok(t, return_tensors="pt", truncation=True, max_length=128))
            ms.append((time.perf_counter() - s) * 1000)
    return np.percentile(ms, 50), np.percentile(ms, 95)

for mid in ["xlm-roberta-base", "distilbert-base-multilingual-cased"] * 2:   # alternate, twice
    p50, p95 = bench(mid)
    print(f"{mid}: p50 {p50:.0f} ms | p95 {p95:.0f} ms")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


xlm-roberta-base: p50 112 ms | p95 329 ms


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

... (10 outputs left out)

xlm-roberta-base: p50 113 ms | p95 342 ms


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilbert-base-multilingual-cased: p50 65 ms | p95 208 ms
